In [1]:
import torch
from torch import nn, optim
import numpy as np
import pandas as pd
import collections
import statistics
import time
import os
import psutil
from river import base, stats, stream, utils, drift, metrics, preprocessing

In [2]:
# --- FUNÇÃO PARA ROTAÇÃO (POV) ---
def generate_rotation_matrix(n_features):
    random_matrix = torch.randn(n_features, n_features)
    q, _ = torch.linalg.qr(random_matrix)
    return q

In [3]:
# --- CLASSE DA REDE NEURAL BLINDADA ---
class FlexibleNeuralNetwork(nn.Module):
    def __init__(self, n_features, n_classes, hidden_layers=[32], use_cnn=False, projection_matrix=None):
        super().__init__()
        self.n_features = n_features
        self.use_cnn = use_cnn
        
        if projection_matrix is not None:
            self.register_buffer('projection', projection_matrix.to(torch.float32))
        else:
            self.projection = None
            
        if use_cnn:
            self.cnn_block = nn.Sequential(
                nn.Conv1d(in_channels=1, out_channels=8, kernel_size=3, padding=1),
                nn.ReLU(),
                nn.Flatten()
            )
            self.in_dim = 8 * n_features
        else:
            self.cnn_block = None
            self.in_dim = n_features

        layers = []
        curr = self.in_dim
        for h in hidden_layers:
            layers.append(nn.Linear(curr, h))
            layers.append(nn.ReLU())
            curr = h
        layers.append(nn.Linear(curr, n_classes))
        self.mlp_head = nn.Sequential(*layers)

    def forward(self, x):
        if x.dim() == 1: x = x.unsqueeze(0)
        x = x.to(torch.float32)
        
        # Força o x a ter exatamente n_features colunas
        if x.shape[1] != self.n_features:
            tmp = torch.zeros((x.shape[0], self.n_features), device=x.device)
            cols = min(x.shape[1], self.n_features)
            tmp[:, :cols] = x[:, :cols]
            x = tmp

        if self.projection is not None:
            x = torch.matmul(x, self.projection)
        
        if self.use_cnn:
            x = x.unsqueeze(1)
            x = self.cnn_block(x)
        
        return self.mlp_head(x)

In [4]:
import collections
from river import base, drift, utils

class HeterogeneousADWINBagging(base.Ensemble, base.Classifier):
    def __init__(self, models: list, seed: int = 42):
        super().__init__(models=models)
        self.seed = seed
        self._rng = np.random.RandomState(seed)
        
        # Cada modelo da lista terá seu próprio detector ADWIN
        self._detectors = [drift.ADWIN(delta=1e-3) for _ in range(len(models))]
        self._total_drifts = 0

    def learn_one(self, x, y):
        for i, model in enumerate(self.models):
            # O Bagging Online usa a distribuição de Poisson para pesos
            k = self._rng.poisson(1)
            if k > 0:
                for _ in range(k):
                    model.learn_one(x, y)

            # Test-then-train para o ADWIN (monitora o erro do modelo)
            y_pred = model.predict_one(x)
            error = 0 if y == y_pred else 1
            
            self._detectors[i].update(error)
            
            if self._detectors[i].drift_detected:
                self._total_drifts += 1
                # Reseta o modelo preservando sua arquitetura original
                self.models[i] = model.clone()
                self._detectors[i] = drift.ADWIN()
        return self

    def predict_proba_one(self, x):
        probas = collections.Counter()
        for model in self.models:
            p = model.predict_proba_one(x)
            for cls, val in p.items():
                probas[cls] += val / len(self.models)
        return probas

In [5]:
from deep_river import classification

device = 'cuda' if torch.cuda.is_available() else 'cpu'
n_feat = 17 # Fixamos em 10 conforme o erro observado
n_classes = 2

perfil_configs = [
    {"use_cnn": False, "opt": optim.SGD,  "lr": 0.1,   "proj": False, "layers": [32]},
    {"use_cnn": False, "opt": optim.Adam, "lr": 0.001, "proj": True,  "layers": [64, 32]},
    {"use_cnn": True,  "opt": optim.SGD,  "lr": 0.05,  "proj": False, "layers": [32]},
    {"use_cnn": True,  "opt": optim.Adam, "lr": 0.001, "proj": True,  "layers": [32]}
]

ensemble_list = []
for i in range(50):
    cfg = perfil_configs[i % len(perfil_configs)]
    proj_matrix = generate_rotation_matrix(n_feat) if cfg["proj"] else None
    
    m = classification.Classifier(
        module=FlexibleNeuralNetwork(
            n_features=n_feat, 
            n_classes=n_classes, 
            use_cnn=cfg["use_cnn"], 
            projection_matrix=proj_matrix,
            hidden_layers=cfg["layers"]
        ),
        loss_fn=nn.CrossEntropyLoss(),
        optimizer_fn=cfg["opt"],
        lr=cfg["lr"],
        is_feature_incremental=False, 
        device=device
    )
    ensemble_list.append(m)

model_hab = HeterogeneousADWINBagging(models=ensemble_list)
metric = metrics.Accuracy()

In [6]:
file_path = os.path.expanduser("~/moa/aldopaim/AdaptiveRegularizedEnsemble/datasets/elecNormNew.arff")
target_column = "class"
oh_encoder = preprocessing.OneHotEncoder()
scaler = preprocessing.StandardScaler()

# Supondo que dataset_stream, oh_encoder e scaler já existam
batch_size = 32
X_batch, y_batch = [], []
latencies = []
label_map = {}

dataset_stream = stream.iter_arff(file_path, target=target_column)

for count, (x, y) in enumerate(dataset_stream, 1):
    if y not in label_map: label_map[y] = len(label_map)
    X_batch.append(x)
    y_batch.append(label_map[y])
    
    if len(X_batch) == batch_size:
        batch_start = time.perf_counter()
        
        # Pré-processamento
        X_df = pd.DataFrame(X_batch)
        oh_encoder.learn_many(X_df)
        X_oh = oh_encoder.transform_many(X_df).astype(np.float64)
        if hasattr(X_oh, "sparse"): X_oh = X_oh.sparse.to_dense()
        
        scaler.learn_many(X_oh)
        X_scaled = scaler.transform_many(X_oh)
        
        # --- PADDING ROBUSTO PARA n_feat=10 ---
        data_fixed = np.zeros((batch_size, n_feat))
        cols_to_copy = min(X_scaled.shape[1], n_feat)
        data_fixed[:, :cols_to_copy] = X_scaled.iloc[:, :cols_to_copy].values
        
        X_final = pd.DataFrame(data_fixed, columns=[f"f_{j}" for j in range(n_feat)])
        X_dicts = X_final.to_dict(orient='records')

        # --- LOOP DE PROCESSAMENTO COM INSTRUMENTAÇÃO ---
        for i in range(batch_size):
            # 1. Tempo de Predição
            t0 = time.perf_counter()
            y_pred = model_hab.predict_one(X_dicts[i])
            t_pred = (time.perf_counter() - t0) * 1000

            if y_pred is not None:
                metric.update(y_batch[i], y_pred)
            
            # 2. Tempo de Treino e Drift
            t1 = time.perf_counter()
            model_hab.learn_one(X_dicts[i], y_batch[i])
            t_learn = (time.perf_counter() - t1) * 1000

        #total_drifts = sum(m['drift_count'] for m in model_hab._ensemble_members)
        total_drifts = model_hab._total_drifts

        latencies.append((time.perf_counter() - batch_start) * 1000)
        X_batch, y_batch = [], []
        
        # --- MONITORAMENTO DE RECURSOS ---
        if count % 2000 < batch_size:
            process = psutil.Process(os.getpid())
            ram_mb = process.memory_info().rss / (1024 * 1024)
            
            # Detalhes da GPU (Essencial para diagnosticar lentidão)
            vram_alloc = 0
            if torch.cuda.is_available():
                vram_alloc = torch.cuda.memory_allocated() / (1024 * 1024)
                # Dica: Limpar cache periodicamente pode ajudar na latência
                torch.cuda.empty_cache() 

            print(f"Inst: {count:<6} | Acc: {metric.get():.2%} | Drifts: {total_drifts}")
            print(f"LATÊNCIA DETALHADA: Pred: {t_pred:.2f}ms | Learn: {t_learn:.2f}ms")
            print(f"RECURSOS: RAM: {ram_mb:.0f}MB | VRAM: {vram_alloc:.0f}MB")
            print("-" * 80)
            
        # # Processamento
        # for i in range(batch_size):
        #     y_pred = model_arte.predict_one(X_dicts[i])
        #     if y_pred is not None:
        #         metric.update(y_batch[i], y_pred)
        #     model_arte.learn_one(X_dicts[i], y_batch[i])

        # total_drifts = sum(m['drift_count'] for m in model_arte._ensemble_members)
        # latencies.append((time.perf_counter() - batch_start) * 1000)
        # X_batch, y_batch = [], []
        
        # if count % 2000 < batch_size:
        #     avg_lat = (sum(latencies[-20:]) / 20) / batch_size
        #     print(f"Inst: {count:<6} | Acc: {metric.get():.2%} | Lat: {avg_lat:.2f}ms | Drifts: {total_drifts}")

Inst: 2016   | Acc: 70.44% | Drifts: 70
LATÊNCIA DETALHADA: Pred: 22.57ms | Learn: 89.54ms
RECURSOS: RAM: 960MB | VRAM: 20MB
--------------------------------------------------------------------------------
Inst: 4000   | Acc: 69.70% | Drifts: 170
LATÊNCIA DETALHADA: Pred: 15.91ms | Learn: 99.45ms
RECURSOS: RAM: 968MB | VRAM: 21MB
--------------------------------------------------------------------------------
Inst: 6016   | Acc: 69.38% | Drifts: 255
LATÊNCIA DETALHADA: Pred: 19.82ms | Learn: 87.40ms
RECURSOS: RAM: 973MB | VRAM: 21MB
--------------------------------------------------------------------------------
Inst: 8000   | Acc: 68.39% | Drifts: 317
LATÊNCIA DETALHADA: Pred: 15.65ms | Learn: 92.35ms
RECURSOS: RAM: 977MB | VRAM: 21MB
--------------------------------------------------------------------------------
Inst: 10016  | Acc: 67.87% | Drifts: 335
LATÊNCIA DETALHADA: Pred: 21.29ms | Learn: 86.33ms
RECURSOS: RAM: 980MB | VRAM: 21MB
-----------------------------------------------

KeyboardInterrupt: 